# Phase 0 smoke test
Proves Ollama, FLUX.1 schnell (NF4), Real-ESRGAN, rembg and OpenCLIP all run on this Kaggle image.
If the Ollama install fails, log the exact error in DECISIONS.md and stop.

## 1. Settings
Accelerator: **GPU T4 x2** (preferred). Settings → **Internet: On**.
Optional: add an `HF_TOKEN` Kaggle Secret (never paste tokens in the notebook).

In [ ]:
import os
import shutil
import subprocess

# --- edit these ---------------------------------------------------------
REPO_URL   = "https://github.com/<you>/redbubble-pipeline.git"  # or attach the repo as a Kaggle Dataset
DATASET    = "/kaggle/input/redbubble-pipeline"                  # used instead of git if it exists
REPO       = "/kaggle/working/redbubble-pipeline"
NICHE      = "cozy mushroom cottage"
N_DESIGNS  = 10
# ------------------------------------------------------------------------
if not os.path.exists(REPO):
    if os.path.isdir(DATASET):
        shutil.copytree(DATASET, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print(os.getcwd())

## 2. Install pinned Python dependencies

In [ ]:
!pip install -q -r requirements.txt -r requirements-gpu.txt
!pip install -q -e . --no-deps

## 3. Install and start Ollama, pull the text model
Dual T4: Ollama runs on GPU 0 with `qwen2.5:14b`; the pipeline process uses GPU 1.
Single GPU: `qwen2.5:7b`, and the runner unloads it (keep_alive 0) before FLUX loads.

In [ ]:
from pipeline.notebook import install_ollama, load_kaggle_secrets, pull_model, start_ollama
from pipeline.utils.gpu import count_gpus

print("secrets loaded:", load_kaggle_secrets())
N_GPUS = count_gpus()
TEXT_MODEL = "qwen2.5:14b" if N_GPUS >= 2 else "qwen2.5:7b"
print(f"{N_GPUS} GPU(s) -> {TEXT_MODEL}")
install_ollama()          # if this fails: copy the error into DECISIONS.md and stop (spec section 4)
start_ollama(gpu="0")
pull_model(TEXT_MODEL)
P = f"pipeline --config {REPO}/config.yaml --set llm.text_model={TEXT_MODEL}"

In [ ]:
!{P} init
!python scripts/smoke_test.py --config config.yaml

In [ ]:
from IPython.display import Image as Show
from IPython.display import display

display(Show("/kaggle/working/smoke/flux.png", width=384))
display(Show("/kaggle/working/smoke/cutout.png", width=384))
with open("/kaggle/working/smoke/report.json") as fh:
    print(fh.read()[:3000])

Also run the CPU test-suite on Kaggle once:

In [ ]:
!pip install -q -r requirements-dev.txt
!python -m pytest -q -m 'not slow'